# 🏙️ Analista Virtual — Mercado Imobiliário de SP (versão Google Colab)

Rode as células **em ordem**, clicando no ▶️ de cada uma.

### Antes de começar (uma vez só)
1. No seu **Google Drive**, crie uma pasta chamada **`ProjetoIA`** e coloque nela estes 3 arquivos:
   - `projeto_analista_virtual.zip`
   - as planilhas do ITBI (`.xlsx`), uma por ano (2019 a 2025)
2. Crie uma chave **grátis** do Groq em **https://console.groq.com/keys** → *Create API Key*.
3. Aqui no Colab, clique no ícone de **chave 🔑 (Secrets)** na barra lateral esquerda → *Add new secret*:
   - Nome: `GROQ_API_KEY`
   - Valor: a chave que você copiou
   - Ative o botão **"Notebook access"**.

> A chave fica nos *Secrets* do Colab, nunca dentro do código — é o guardrail "chave de API fora do repositório".

## 1. Conectar o Google Drive e preparar a pasta do projeto

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, zipfile, glob
PASTA_DRIVE = '/content/drive/MyDrive/ProjetoIA'

assert os.path.exists(PASTA_DRIVE), f'Crie a pasta {PASTA_DRIVE} no Drive com o zip e as planilhas.'
with zipfile.ZipFile(f'{PASTA_DRIVE}/projeto_analista_virtual.zip') as z:
    z.extractall('/content')

os.makedirs('/content/projeto/data/raw', exist_ok=True)
planilhas = glob.glob(f'{PASTA_DRIVE}/*.xlsx')
for p in planilhas:
    shutil.copy(p, '/content/projeto/data/raw/')
print('Planilhas copiadas:', [os.path.basename(p) for p in planilhas])
assert len(planilhas) >= 1, 'Coloque as planilhas do ITBI (.xlsx) na pasta ProjetoIA.'

%cd /content/projeto

## 2. Instalar as bibliotecas (≈1 min)

In [ ]:
!pip install -q openai python-calamine python-dotenv "gradio>=5,<6"

## 3. Carregar os dados e rodar os testes de qualidade (≈1–3 min, conforme o número de anos)
Junta as 24 abas mensais, limpa, monta o modelo **fato + dimensões** no SQLite e roda os testes
(nulos, duplicatas, chaves órfãs). Todas as linhas devem terminar com **[OK]**.

In [ ]:
!python src/load_data.py

## 4. Testes dos guardrails (somente leitura, LIMIT, timeout…) — não usa a chave

In [ ]:
!python -m pytest -q tests/

## 5. Configurar o modelo de linguagem
Usa o **Groq** (grátis, sem cartão) com o modelo `openai/gpt-oss-120b`.
O plano grátis tem limite de tokens por minuto e por dia: o sistema espera sozinho quando bate o limite por minuto.

In [ ]:
from google.colab import userdata
os.environ['LLM_API_KEY']          = userdata.get('GROQ_API_KEY')
os.environ['LLM_BASE_URL']         = 'https://api.groq.com/openai/v1'
os.environ['LLM_MODEL']            = 'openai/gpt-oss-120b'
os.environ['LLM_REASONING_EFFORT'] = 'low'
print('Modelo configurado:', os.environ['LLM_MODEL'])

## 6. Primeira pergunta (teste rápido)

In [ ]:
import sys
sys.path.insert(0, '/content/projeto/src')
from orchestrator import AnalistaVirtual
import pandas as pd
from IPython.display import Markdown, display

analista = AnalistaVirtual()

def perguntar(pergunta):
    r = analista.responder(pergunta)
    display(Markdown(f'**[{r.classificacao}]** · {r.tempo_s}s · {r.tentativas_sql} execução(ões) de SQL\n\n' + r.texto))
    if r.linhas:
        display(pd.DataFrame(r.linhas, columns=r.colunas).head(20))
    if r.sql:
        print('SQL usado:\n' + r.sql)
    return r

r = perguntar('Quantas vendas tivemos em 2025?')

Teste outras perguntas mudando o texto abaixo:

In [ ]:
r = perguntar('Qual o preço médio do m² por zona em 2025?')

## 7. Interface (Gradio)
Gera um link público `https://....gradio.live` que funciona enquanto este notebook estiver aberto —
dá para usar na apresentação. Para parar, clique no ⏹️ da célula.

In [ ]:
from app_gradio import criar_interface
criar_interface().launch(share=True, debug=False)

## 8. Avaliação com o golden set — rodada **"antes"**
Roda as 20 perguntas do `eval/golden_set.csv` e compara com as respostas de referência.
Leva uns 20–30 minutos. Os resultados são salvos a cada pergunta.
Se a **cota diária** acabar no meio, o script para e avisa: no dia seguinte rode a mesma célula —
o `--continuar` pula as perguntas que já foram feitas.

In [ ]:
!python eval/run_eval.py --rodada antes --continuar

Ver os resultados pergunta a pergunta:

In [ ]:
res = pd.read_csv('eval/resultados.csv')
res[['rodada','id','nivel','acertou','motivo','tentativas_sql','tempo_s']]

## 9. Salvar os resultados no Drive
O Colab apaga os arquivos quando a sessão fecha. Esta célula copia os resultados para `ProjetoIA/resultados` no Drive.

In [ ]:
destino = f'{PASTA_DRIVE}/resultados'
os.makedirs(destino, exist_ok=True)
for f in ['eval/resultados.csv', 'eval/resumo_metricas.md', 'data/processed/relatorio_qualidade.txt']:
    if os.path.exists(f):
        shutil.copy(f, destino)
print('Copiado para', destino, ':', os.listdir(destino))

## 10. Depois das correções — rodada **"depois"**
Depois de analisar os erros (`eval/analise_erros.md`) e ajustar a camada semântica/prompts,
rode a avaliação de novo. A rodada "antes" continua salva no mesmo CSV para comparação.

> Se você editar arquivos aqui no Colab (painel 📁 à esquerda → duplo clique no arquivo),
> rode a célula da seção 6 de novo para recarregar o sistema.

In [ ]:
!python eval/run_eval.py --rodada depois --continuar